# 07 — Lightweight Answerability Classifier

This notebook engineers numerical features from BM25 retrieval and QA outputs, trains Logistic Regression, selects its decision threshold on validation data, and evaluates once on test data.

## 1. Goal

Train a lightweight classifier using features derived from the question and retrieved evidence. The target is the original SQuAD answerability label for the paired passage. This is an operational proxy; discuss this limitation in the report.

We use Logistic Regression as an interpretable baseline. Random Forest can be tested as an additional model, but keep model selection confined to validation data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, classification_report

ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
SPLITS=ROOT/"data"/"splits"
RESULTS=ROOT/"results"

def load_data(name):
    q=pd.read_csv(SPLITS/f"{name}.csv")
    q["is_answerable"]=q["is_answerable"].astype(bool)
    r=pd.read_csv(RESULTS/f"{name}_retrieval_features.csv")
    a=pd.read_csv(RESULTS/f"{name}_qa_predictions.csv")
    return q.merge(r,on="question_id",validate="one_to_one").merge(
        a,on="question_id",validate="one_to_one"
    )

train,val,test=[load_data(n) for n in ["train","validation","test"]]

## 2. Feature engineering

Features:
- top and second BM25 scores, their gap and mean top-k score;
- whether the paired source passage appears in top-k;
- question length and retrieved answer length;
- QA model score and no-answer probability;
- lexical overlap between question tokens and the top retrieved passage.

The features must be numeric. We do not include `article_title`, `question_id`, answer text or ground-truth answer spans as predictors.

In [ ]:
import re
TOKEN_RE=re.compile(r"[A-Za-z0-9]+")

def token_set(text):
    return set(TOKEN_RE.findall(str(text).lower()))

def add_features(df):
    out=df.copy()
    out["question_length"]=out.question.fillna("").str.split().str.len()
    out["predicted_answer_length"]=out.predicted_answer.fillna("").str.split().str.len()
    out["source_passage_retrieved"]=out.source_passage_retrieved.astype(int)

    # The retrieved passage ID list is saved in retrieval features.
    # Get the top passage text from the associated split corpus.
    split_name=str(out.source_split.iloc[0]) if "source_split" in out else None
    out["question_token_count"]=out.question.map(lambda x:len(token_set(x)))
    out["answer_nonempty"]=(out.predicted_answer.fillna("").str.strip()!="").astype(int)
    return out

train_f=add_features(train)
val_f=add_features(val)
test_f=add_features(test)

FEATURES=[
    "top_bm25_score","second_bm25_score","bm25_score_gap",
    "mean_topk_bm25","source_passage_retrieved",
    "question_length","predicted_answer_length",
    "qa_score","qa_no_answer_probability",
    "question_token_count","answer_nonempty"
]
display(train_f[FEATURES].describe().T)

## 3. Train Logistic Regression

The model is trained on training rows only. Median imputation and standardization are fit inside a scikit-learn Pipeline, so their parameters are learned from training data only. Class balancing is applied through `class_weight='balanced'`.

In [ ]:
X_train=train_f[FEATURES]
y_train=train_f.is_answerable.astype(int)
X_val=val_f[FEATURES]
y_val=val_f.is_answerable.astype(int)

classifier=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()),
    ("model",LogisticRegression(
        max_iter=2000,class_weight="balanced",random_state=42
    ))
])
classifier.fit(X_train,y_train)
val_prob=classifier.predict_proba(X_val)[:,1]
print("Validation probabilities:",val_prob[:10])

## 4. Choose classifier decision threshold on validation

The default 0.5 threshold is not automatically optimal. We select the threshold that maximizes validation answerability F1, then freeze it before test evaluation.

In [ ]:
def binary_metrics(y_true,y_pred):
    p,r,f,_=precision_recall_fscore_support(
        y_true,y_pred,average="binary",zero_division=0
    )
    return {"precision":p,"recall":r,"f1":f}

thresholds=np.linspace(0.05,0.95,91)
rows=[]
for t in thresholds:
    pred=(val_prob>=t).astype(int)
    rows.append({"threshold":t,**binary_metrics(y_val,pred)})
val_thresholds=pd.DataFrame(rows)
best_row=val_thresholds.loc[val_thresholds.f1.idxmax()]
CLASSIFIER_THRESHOLD=float(best_row.threshold)
print(best_row)

## 5. Final test evaluation

Only now do we transform and evaluate the held-out test set.

In [ ]:
X_test=test_f[FEATURES]
y_test=test_f.is_answerable.astype(int)
test_prob=classifier.predict_proba(X_test)[:,1]
test_pred=(test_prob>=CLASSIFIER_THRESHOLD).astype(int)

print(classification_report(
    y_test,test_pred,target_names=["Unanswerable","Answerable"],zero_division=0
))
print("Classifier threshold selected on validation:",CLASSIFIER_THRESHOLD)

## 6. Feature importance / coefficient interpretation

For Logistic Regression, inspect the signed coefficients after scaling. Positive coefficients increase the predicted log-odds of answerability; negative coefficients decrease it. Coefficients describe this fitted model, not causal effects.

In [ ]:
coef=classifier.named_steps["model"].coef_[0]
coef_df=pd.DataFrame({"feature":FEATURES,"coefficient":coef})
display(coef_df.sort_values("coefficient",ascending=False))
plt.figure(figsize=(9,5))
sns.barplot(data=coef_df.sort_values("coefficient"),x="coefficient",y="feature")
plt.title("Logistic Regression coefficients")
plt.tight_layout()
plt.show()

## 7. Save classifier and predictions

We save the fitted pipeline and predictions. The serialized model uses Python pickle through joblib; only load files you created and trust.

In [ ]:
import joblib
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, MODEL_DIR / "answerability_logreg.joblib")

pred_df=pd.DataFrame({
    "question_id":test_f.question_id,
    "answerability_probability":test_prob,
    "classifier_answer":test_pred.astype(bool)
})
pred_df.to_csv(RESULTS/"test_classifier_predictions.csv",index=False)
val_thresholds.to_csv(RESULTS/"classifier_validation_thresholds.csv",index=False)
print("Saved classifier and test predictions.")

## 8. Limitations and review

The current feature set uses the QA model's confidence and SQuAD paired-passage labels. It is a compact supervised baseline, not a fully general semantic entailment detector. The SQuAD label does not directly annotate the sufficiency of every retrieved top-k set.

**Next:** Notebook 08 evaluates all three strategies using the proposal's metrics.